<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/milmor/NLP/blob/main/Notebooks/20_FAISS_hf.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
</table>

# FAISS  
- Game of thrones book: https://www.kaggle.com/datasets/khulasasndh/game-of-thrones-books
- Harry Potter book: https://www.kaggle.com/datasets/gulsahdemiryurek/harry-potter-dataset

In [1]:
#!pip install -q sentence-transformers faiss-cpu

In [2]:
import torch
import pandas as pd
from datasets import Dataset
import sentence_transformers

torch.__version__, sentence_transformers.__version__

('2.7.1+cu126', '5.4.1')

## 1.- Dataset

In [3]:
path = './Book1.txt'
book = open(path, 'rb').read().decode(encoding='utf-8').lower()

print(f'Words: {len(book)}')

Words: 474429


In [4]:
import re

words = re.findall(r'\b\w+\b|[\.,;!?()"\']', book)

maxlen = 50
# Crear lotes de 50 palabras
sentences = [words[i:i + maxlen] for i in range(0, len(words), maxlen)]

In [5]:
sentences[0][:20]

['the',
 'boy',
 'who',
 'lived',
 'mr',
 '.',
 'and',
 'mrs',
 '.',
 'dursley',
 ',',
 'of',
 'number',
 'four',
 ',',
 'privet',
 'drive',
 ',',
 'were',
 'proud']

In [6]:
len(sentences)

1968

In [7]:
my_dict = {
    "id": list(range(len(sentences))),
    "text": [" ".join(sentence) for sentence in sentences]
}

dataset = Dataset.from_dict(my_dict)
dataset

Dataset({
    features: ['id', 'text'],
    num_rows: 1968
})

In [8]:
dataset[0]

{'id': 0,
 'text': 'the boy who lived mr . and mrs . dursley , of number four , privet drive , were proud to say that they were perfectly normal , thank you very much . they were the last people you d expect to be involved in anything strange or mysterious ,'}

## 2.- Model

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2" # Using a small model
)

In [10]:
def embed(batch): # adds embedding column
    information = batch["text"]
    return {"embeddings" : model.encode(information)}

dataset = dataset.map(embed,batched=True,batch_size=16)

Map:   0%|          | 0/1968 [00:00<?, ? examples/s]

In [11]:
dataset

Dataset({
    features: ['id', 'text', 'embeddings'],
    num_rows: 1968
})

In [12]:
len(dataset[0]['embeddings'])

384

In [13]:
data = dataset.add_faiss_index("embeddings")

  0%|          | 0/2 [00:00<?, ?it/s]

## 3.- Search

In [14]:
def search(query: str, k: int = 3):
    embedded_query = model.encode(query) # embed new query
    scores, retrieved_examples = data.get_nearest_examples( 
        "embeddings", embedded_query, # compare our new embedded query with the dataset embeddings
        k=k # get only top k results
    )
    return scores, retrieved_examples

In [15]:
scores, result = search("harry loves", 5) 
result['text']

['were all quite happy to join in dudley s favorite sport harry hunting . this was why harry spent as much time as possible out of the house , wandering around and thinking about the end of the holidays , where he could see a tiny ray of hope .',
 'harry had the best morning he d had in a long time . he was careful to walk a little way apart from the dursleys so that dudley and piers , who were starting to get bored with the animals by lunchtime , wouldn t fall back on their favorite',
 '. page 274 harry potter and the philosophers stone j . k . rowling from being one of the most popular and admired people at the school , harry was suddenly the most hated . even ravenclaws and hufflepuffs turned on him , because everyone had been longing to see',
 'cats , and she didn t seem quite as fond of them as before . she let harry watch television and gave him a bit of chocolate cake that tasted as though she d had it for several years . that evening , dudley paraded around the living room for'

In [16]:
scores, result = search("ron loves", 5) 
result['text']

['! only ron stood by him . they ll all forget this in a few weeks . fred and george have lost loads of points in all the time they ve been here , and people still like them . they ve never lost a hundred and fifty points in',
 'most of their free time in the library with her , trying to get through all their extra work . i ll never remember this , ron burst out one afternoon , throwing down his quill and looking longingly out of the library window . it was the first really',
 'stepped aside , but with ron in front of the mirror , he couldn t see his family anymore , just ron in his paisley pajamas . ron , though , was staring transfixed at his image . look at me ! he said . can you see all your',
 'd listen about the time he d almost hit a hang glider on charlie s old broom . everyone from wizarding families talked about quidditch constantly . ron had already had a big argument with dean thomas , who shared their dormitory , about soccer . ron couldn t see',
 'to catch me ? if he find

In [17]:
scores

array([0.96991444, 0.9813764 , 0.9865732 , 0.9965582 , 0.99761367],
      dtype=float32)